In [ ]:
import duckdb
from pathlib import Path

# Path to the DuckDB database file
DATA_DIR = Path("../data/raw")

# Connect to the DuckDB database
con = duckdb.connect("../northwind.duckdb")

In [4]:
list(DATA_DIR.glob("*.csv"))

[WindowsPath('../data/raw/categories.csv'),
 WindowsPath('../data/raw/customers.csv'),
 WindowsPath('../data/raw/employees.csv'),
 WindowsPath('../data/raw/orders.csv'),
 WindowsPath('../data/raw/order_details.csv'),
 WindowsPath('../data/raw/products.csv'),
 WindowsPath('../data/raw/shippers.csv')]

In [ ]:
con.sql("""
SELECT *
FROM read_csv_auto('../data/raw/categories.csv')
"""
).df() 

,categoryID,categoryName,description
0,1,Beverages,"Soft drinks, coffees, teas, beers, and ales"
1,2,Condiments,"Sweet and savory sauces, relishes, spreads, an..."
2,3,Confections,"Desserts, candies, and sweet breads"
3,4,Dairy Products,Cheeses
4,5,Grains & Cereals,"Breads, crackers, pasta, and cereal"
5,6,Meat & Poultry,Prepared meats
6,7,Produce,Dried fruit and bean curd
7,8,Seafood,Seaweed and fish


In [40]:
con.sql("""
SELECT *
FROM read_csv_auto(
    '../data/raw/customers.csv',
    encoding = 'latin-1'
);
""").show()

┌────────────┬────────────────────────────────────┬─────────────────────────┬───────────────────────────┬─────────────┬─────────┐
│ customerID │            companyName             │       contactName       │       contactTitle        │    city     │ country │
│  varchar   │              varchar               │         varchar         │          varchar          │   varchar   │ varchar │
├────────────┼────────────────────────────────────┼─────────────────────────┼───────────────────────────┼─────────────┼─────────┤
│ ALFKI      │ Alfreds Futterkiste                │ Maria Anders            │ Sales Representative      │ Berlin      │ Germany │
│ ANATR      │ Ana Trujillo Emparedados y helados │ Ana Trujillo            │ Owner                     │ Mexico City │ Mexico  │
│ ANTON      │ Antonio Moreno Taquería            │ Antonio Moreno          │ Owner                     │ Mexico City │ Mexico  │
│ AROUT      │ Around the Horn                    │ Thomas Hardy            │ Sales Repres

In [41]:
# Create tables in DuckDB from the CSV files
con.sql("""
CREATE OR REPLACE TABLE customers AS
SELECT *
FROM read_csv('../data/raw/customers.csv',
        encoding = 'latin-1'
        );
""")

con.sql("""
CREATE OR REPLACE TABLE categories AS
SELECT *
FROM read_csv_auto('../data/raw/categories.csv');
""")

con.sql("""
CREATE OR REPLACE TABLE employees AS
SELECT *
FROM read_csv_auto('../data/raw/employees.csv');
""")

con.sql("""
CREATE OR REPLACE TABLE order_details AS
SELECT *
FROM read_csv_auto('../data/raw/order_details.csv');
""")

con.sql("""
CREATE OR REPLACE TABLE orders AS
SELECT *
FROM read_csv_auto('../data/raw/orders.csv');
""")

con.sql("""
CREATE OR REPLACE TABLE products AS
SELECT *
FROM read_csv_auto('../data/raw/products.csv', encoding = 'latin-1');
""")

con.sql("""
CREATE OR REPLACE TABLE shippers AS
SELECT *
FROM read_csv_auto('../data/raw/shippers.csv');
""")

In [42]:
#check that the tables were created successfully
con.sql("""
    SHOW TABLES; """  ).df()
        

,name
0,categories
1,customers
2,employees
3,order_details
4,orders
5,products
6,shippers


In [43]:
from IPython.display import display

# Check the number of records in each table
display(con.sql("""
        SELECT COUNT(*) as category_count FROM categories;""").df())

display(con.sql("""
        SELECT COUNT(*) as customer_count FROM customers;""").df())

display(con.sql("""
        SELECT COUNT(*) as employee_count FROM employees;""").df())

display(con.sql("""
        SELECT COUNT(*) as order_details_count FROM order_details;""").df())

display(con.sql("""
        SELECT COUNT(*) as orders_count FROM orders;""").df())

display(con.sql("""
        SELECT COUNT(*) as product_count FROM products;""").df())

display(con.sql("""
        SELECT COUNT(*) as shippers_count FROM shippers;""").df())


,category_count
0,8


,customer_count
0,91


,employee_count
0,9


,order_details_count
0,2155


,orders_count
0,830


,product_count
0,77


,shippers_count
0,3


In [44]:
# Generate markdown documentation for the table schemas

from pathlib import Path
# Path to the output markdown file
output_file = Path("../observations.md")

tables = [i for i in con.sql("SHOW TABLES;").df()["name"]]

with output_file.open("a", encoding="utf-8") as f:
    f.write("\n\n# Table Schemas\n\n")

    for table in tables:
        schema_df = con.sql(f"""
            DESCRIBE {table};
        """).df()

        f.write(f"## Schema for table: `{table}`\n\n")
        f.write(schema_df.to_markdown(index=False))
        f.write("\n\n")

In [45]:
from pathlib import Path

observations_file = Path("../observations.md")
data_dictionary_file = Path("../data/data_dictionary.csv")

df = con.sql(f"""
       SELECT * FROM read_csv_auto('{data_dictionary_file}', normalize_names=True);""").df()

md_table = df.to_markdown(index=False)
with observations_file.open("a", encoding="utf-8") as f:
    f.write("\n\n# Data Dictionary\n\n")
    f.write(md_table)
    

In [ ]:
con.sql("""
SELECT
    shipperID,
    COUNT(*) AS row_count
FROM shippers
GROUP BY shipperID
HAVING COUNT(*) > 1;
""").df()

#checked for uniqness for the Primary Key for all tables

,shipperID,row_count


In [ ]:
# order_details has a composite primary key of orderID and productID, so we need to check for duplicates based on both columns together

con.sql("""

SELECT
    orderID,
    productID,
    COUNT(*) AS row_count
FROM order_details
GROUP BY orderID, productID
HAVING COUNT(*) > 1;
        """)

┌─────────┬───────────┬───────────┐
│ orderID │ productID │ row_count │
│  int64  │   int64   │   int64   │
└─────────┴───────────┴───────────┘
              0 rows             

In [13]:
from IPython.display import display, Markdown

# Checks orders.customerID → customers.customerID
display(
con.sql("""
  SELECT 
    o.customerID,
    COUNT(*) AS unmatched_orders
FROM orders o
LEFT JOIN customers c 
    ON o.customerID = c.customerID
WHERE c.customerID IS NULL
GROUP BY o.customerID;
      
        """).df()
)

# Checks orders.employeeID → employees.employeeID
display(
    con.sql("""
       SELECT  e.employeeID, COUNT(*) as NullValues FROM orders o
            LEFT JOIN employees e ON o.employeeID = e.employeeID
            WHERE e.employeeID IS NULL
            GROUP BY e.employeeID 
            """).df()
)



display(Markdown("### Check: orders.shipperID → shippers.shipperID"))
# Checks orders.shipperID → shippers.shipperID
display(
    con.sql("""
        SELECT 
            o.shipperID AS foreign_key_value,
            COUNT(*) AS unmatched_orders
        FROM orders AS o
        LEFT JOIN shippers AS s 
            ON o.shipperID = s.shipperID
        WHERE s.shipperID IS NULL
        GROUP BY o.shipperID
        ORDER BY unmatched_orders DESC;
    """).df()
)

display(Markdown("### Check: order_details.orderID → orders.orderID"))
# Checks order_details.orderID → orders.orderID
display(
    con.sql("""
        SELECT 
            od.orderID AS foreign_key_value,
            COUNT(*) AS unmatched_order_lines
        FROM order_details AS od
        LEFT JOIN orders AS o 
            ON od.orderID = o.orderID
        WHERE o.orderID IS NULL
        GROUP BY od.orderID
        ORDER BY unmatched_order_lines DESC;
    """).df()
)


display(Markdown("### Check: order_details.productID → products.productID"))
# Checks order_details.productID → products.productID
display(
    con.sql("""
        SELECT 
            od.productID AS foreign_key_value,
            COUNT(*) AS unmatched_order_lines
        FROM order_details AS od
        LEFT JOIN products AS p 
            ON od.productID = p.productID
        WHERE p.productID IS NULL
        GROUP BY od.productID
        ORDER BY unmatched_order_lines DESC;
    """).df()
)

display(Markdown("### Check: products.categoryID → categories.categoryID"))
# Checks products.categoryID → categories.categoryID
display(
    con.sql("""
        SELECT 
            p.categoryID AS foreign_key_value,
            COUNT(*) AS unmatched_products
        FROM products AS p
        LEFT JOIN categories AS c 
            ON p.categoryID = c.categoryID
        WHERE c.categoryID IS NULL
        GROUP BY p.categoryID
        ORDER BY unmatched_products DESC;
    """).df()
)

display(Markdown("### Check: employees.reportsTo → employees.employeeID"))
# Checks employees.reportsTo → employees.employeeID (self-referential foreign key)
display(
    con.sql("""
        SELECT 
            e.reportsTo AS foreign_key_value,
            COUNT(*) AS unmatched_employees
        FROM employees AS e
        LEFT JOIN employees AS manager
            ON e.reportsTo = manager.employeeID
        WHERE manager.employeeID IS NULL
          AND e.reportsTo IS NOT NULL
        GROUP BY e.reportsTo
        ORDER BY unmatched_employees DESC;
    """).df()
)

,customerID,unmatched_orders


,employeeID,NullValues


### Check: orders.shipperID → shippers.shipperID

,foreign_key_value,unmatched_orders


### Check: order_details.orderID → orders.orderID

...

,foreign_key_value,unmatched_order_lines


### Check: products.categoryID → categories.categoryID

,foreign_key_value,unmatched_products


### Check: employees.reportsTo → employees.employeeID

,foreign_key_value,unmatched_employees


In [ ]:
# Number of Orders per Customer

con.sql("""
   SELECT c.customerID, c.contactName, Count(o.orderID) as order_count FROM customers c
        LEFT jOIN orders o ON c.customerID = o.customerID
        GROUP BY c.customerID, c.contactName
        ORDER BY order_count DESC;
        
        """  ).df()

,customerID,contactName,order_count
0,SAVEA,Jose Pavarotti,31
1,ERNSH,Roland Mendel,30
2,QUICK,Horst Kloss,28
3,HUNGO,Patricia McKenna,19
4,FOLKO,Maria Larsson,19
...,...,...,...
86,LAZYK,John Steel,2
87,GROSR,Manuel Pereira,2
88,CENTC,Francisco Chang,1
89,FISSA,Diego Roel,0


In [26]:
# Numbers of products per category

con.sql("""
SELECT c.categoryName, Count(p.productID) as product_count FROM categories c
        LEFT JOIN products p ON 
        c.categoryID = p.categoryID
        GROUP BY c.categoryName
        ORDER BY product_count DESC;
""").df()

,categoryName,product_count
0,Confections,13
1,Beverages,12
2,Condiments,12
3,Seafood,12
4,Dairy Products,10
5,Grains & Cereals,7
6,Meat & Poultry,6
7,Produce,5


In [30]:
# Number of products per order
con.sql("""
SELECT
    o.orderID,
    COUNT(od.productID) AS product_lines,
    SUM(od.quantity) AS total_units_ordered
FROM orders AS o
LEFT JOIN order_details AS od
    ON o.orderID = od.orderID
GROUP BY o.orderID
ORDER BY total_units_ordered DESC;
""").df()

,orderID,product_lines,total_units_ordered
0,10895,4,346.0
1,11030,4,330.0
2,10847,6,288.0
3,10515,5,286.0
4,10678,4,280.0
...,...,...,...
825,10767,1,2.0
826,10531,1,2.0
827,10422,1,2.0
828,10807,1,1.0
